# 10 - Augmentasi Data Negatif

Gabungan dari `01_pengumpulan_data/06_data_negatif_tambahan` dan `01_pengumpulan_data/07_persiapan_data`
(DIGABUNG, bukan 2 tahap terpisah -- dicek ternyata dua folder itu sama-sama cuma
mengoreksi/menambah data ke `dataset_final.csv` hasil tahap 09, bukan proses baru dari nol).

**Kenapa perlu tahap ini:** skor sentimen influencer butuh model yang bisa kenali komentar NEGATIF
dengan baik, tapi dari hasil tahap 09, komentar negatif cuma segelintir dari ~2000 sample (minoritas
ekstrem) -- model akan buruk mendeteksi kelas ini kalau datanya tidak ditambah.

**Alur (1 cell dieksekusi per tahap di bawah):**
1. `search_negative_candidates()` -- Layer 1 (semantic search, embedding bge-m3 via Ollama) + Layer 2
   (LLM judge qwen2.5:7b + few-shot) cari kandidat komentar negatif baru dari `comments.csv`.
2. **[MANUAL, di luar notebook ini]** -- review satu-per-satu kandidat lewat web app
   (`python 02_labeling.py`, buka `http://localhost:5050`) -> `review_manual.csv`.
3. `merge_negative_candidates()` -- cross-check gabungan negatif (sanity check jumlah saja).
4. `apply_review_corrections()` -- koreksi manual 4 comment_id (hasil diskusi ulang).
5. `integrate_into_dataset()` -- tambahkan baris negatif hasil review ke `dataset_final.csv`.
6. `apply_dataset_corrections()` -- koreksi manual kedua (3 teks + 1 substring) di level dataset final.
7. `split_dataset()` -- re-split train 80% / test 20% (stratified) + 5-fold CV di train.
8. `extract_minority_for_synthesis()` -- ambil negatif & positif di TRAIN untuk sintesis paraphrase tahap 11.

**Konsolidasi dari:**
- `01_pengumpulan_data/06_data_negatif_tambahan/01_cari_kandidat_negatif_2.ipynb` -> `search_negative_candidates()`
- `01_pengumpulan_data/06_data_negatif_tambahan/03_negatif_only.py` -> `merge_negative_candidates()`
- `01_pengumpulan_data/07_persiapan_data/02_gabung_data.py` -> `integrate_into_dataset()`
- `01_pengumpulan_data/07_persiapan_data/01_ambil_data.py` -> `apply_dataset_corrections()`
- `01_pengumpulan_data/07_persiapan_data/03_split_data.py` -> `split_dataset()`
- `01_pengumpulan_data/07_persiapan_data/04_ambil_negatif_train.py` -> `extract_minority_for_synthesis()`

**Sengaja tidak diikutkan:**
- `02_labeling.py` (web app Flask review manual) -- interaktif, jalankan file aslinya langsung.
- `04_review_data.py` -- digabung ke `apply_review_corrections()` (koreksinya sama, dibikin idempotent).

**Catatan eksekusi:** `search_negative_candidates()` berat (embedding ~27 ribu komentar unik + LLM
judge 5000 kandidat, bisa berjam-jam). Skip otomatis kalau output sudah ada. Fungsi lain ringan
(pandas saja).

In [ ]:
import csv
import hashlib
import json
import re
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from sklearn.model_selection import StratifiedKFold, train_test_split

# -------------------------------------------------------------------------
# KONFIGURASI PATH
# -------------------------------------------------------------------------
FINAL_OUTPUT_ROOT = Path.cwd() / "output"
OUTPUT_DIR = FINAL_OUTPUT_ROOT / "10_augmentasi_data_negatif"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COMMENTS_RAW = FINAL_OUTPUT_ROOT / "06_scraping_komentar" / "comments.csv"
DATASET_BINARY_BASELINE = FINAL_OUTPUT_ROOT / "09_persiapan_dataset_training" / "binary" / "dataset_final.csv"

EMBED_CACHE_REF = OUTPUT_DIR / "embed_cache_acuan_negatif.csv"
EMBED_CACHE_RAW = OUTPUT_DIR / "embed_cache_comments.csv"
LAYER1_PATH = OUTPUT_DIR / "layer1_kandidat_embedding.csv"
KANDIDAT_FINAL = OUTPUT_DIR / "kandidat_final_untuk_review.csv"
REVIEW_MANUAL = OUTPUT_DIR / "review_manual.csv"              # hasil review manual (IRREPLACEABLE)
KOMENTAR_NEGATIF_GABUNGAN = OUTPUT_DIR / "komentar_negatif_gabungan.csv"
DATASET_FINAL = OUTPUT_DIR / "dataset_final.csv"               # working copy, diupdate tiap langkah
MINORITAS_SINTESIS = OUTPUT_DIR / "minoritas_train_untuk_sintesis.csv"

OLLAMA_HOST = "http://localhost:11434"
EMBED_MODEL = "bge-m3"
JUDGE_MODEL = "qwen2.5:7b"
N_CANDIDATES = 5000
N_FEWSHOT_EACH = 4
CHECKPOINT_EVERY = 100
SEED = 42
RANDOM_SEED = 42
TEST_SIZE = 0.2
N_FOLDS = 5

# Koreksi manual (hasil diskusi, lihat markdown step 4 & 6)
REVIEW_KOREKSI = {
    "7592739646564500242": "bukan_negatif",  # "Knp pabriknya ga ttup aja si..."
    "7592645460424098578": "bukan_negatif",  # "gila jahat banget"
    "7594711847862829845": "bukan_negatif",  # "Cerita tak full...koma...intubate..."
    "7652223662414201620": "bukan_negatif",  # comment soal Messi (OOT)
}
DATASET_KOREKSI_TEKS = {
    "ngeri bgtt": "netral",
    "Liat yt nya sampek abis kak, jangan cuman asal komen di tiktok yg setengah2, ini pasien nya di yt disebut kok "
    "nama dokternya dan rs nya, dan dia juga bawak bukti ctscen nya!!": "netral",
    "Nauzubillah tapi aku gak percaya dia pelakunya sedih ya Alloh kenapa harus begitu": "netral",
}
DATASET_KOREKSI_SUBSTRING = "trust issue ke lambung"
print("Config siap.")

## Fungsi: helper embedding + LLM judge (dipakai STEP 1)

In [ ]:
def _cosine_sim_matrix(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    a_n = a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-8)
    b_n = b / (np.linalg.norm(b, axis=1, keepdims=True) + 1e-8)
    return a_n @ b_n.T


def _hash_text(t: str) -> str:
    return hashlib.md5(t.encode("utf-8")).hexdigest()


def _get_embedding(text: str, retries: int = 3) -> np.ndarray:
    for attempt in range(retries):
        try:
            r = requests.post(f"{OLLAMA_HOST}/api/embeddings",
                               json={"model": EMBED_MODEL, "prompt": text}, timeout=30)
            r.raise_for_status()
            return np.array(r.json()["embedding"], dtype=np.float32)
        except Exception:
            if attempt == retries - 1:
                raise
            time.sleep(1.5 * (attempt + 1))


def _load_embed_cache(path: Path) -> dict:
    cache = {}
    if not path.exists():
        return cache
    with open(path, "r", encoding="utf-8", newline="") as f:
        reader = csv.reader(f)
        next(reader, None)
        for row in reader:
            if len(row) < 3:
                continue
            h, text, emb_json = row[0], row[1], row[2]
            try:
                cache[h] = (text, np.array(json.loads(emb_json), dtype=np.float32))
            except Exception:
                continue
    return cache


def _embed_with_cache(texts: list, cache_path: Path, desc: str) -> dict:
    cache = _load_embed_cache(cache_path)
    result = {text: vec for (text, vec) in cache.values()}
    already_hashed = set(cache.keys())
    todo = [t for t in texts if _hash_text(t) not in already_hashed]
    print(f"[{desc}] cache: {len(texts) - len(todo)}/{len(texts)} sudah ada, {len(todo)} perlu di-embed baru")
    if not todo:
        return result

    is_new_file = not cache_path.exists()
    with open(cache_path, "a", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        if is_new_file:
            writer.writerow(["hash", "text", "embedding_json"])
            f.flush()
        for i, t in enumerate(todo, 1):
            try:
                vec = _get_embedding(t)
            except Exception as e:
                print(f"   Gagal embed (dilewati, akan dicoba lagi kalau dijalankan ulang): {t[:60]!r} -> {e}")
                continue
            result[t] = vec
            writer.writerow([_hash_text(t), t, json.dumps(vec.tolist())])
            f.flush()
            if i % 500 == 0:
                print(f"   [{desc}] {i}/{len(todo)}")
    return result


JUDGE_PROMPT_TEMPLATE = """Kamu menilai apakah komentar TikTok di bawah ini termasuk NEGATIF terhadap konten atau dokter/influencer kesehatan dalam video tersebut.

DEFINISI (penting, baca pelan-pelan):
- NEGATIF = komentar dengan nada negatif (keluhan, kekecewaan, sarkasme, kritik) yang DITUJUKAN ke konten videonya atau ke dokter/influencernya.
- BUKAN NEGATIF = semua yang lain, termasuk: curhat/keluhan pribadi (soal penyakit, hidup, pengalaman sendiri) yang TIDAK menyerang/mengkritik dokter atau kontennya, pertanyaan biasa, pujian, komentar netral/lucu.
- Keluhan pribadi (misal "capek banget", "gerd saya kambuh", "8 tahun saya begini") itu BUKAN NEGATIF meskipun nadanya sedih/kesal, SELAMA tidak menyindir dokter/kontennya.

Contoh dari data yang sudah pernah dilabel manusia:
{fewshot}

Sekarang nilai komentar baru ini:
"{comment}"

Jawab PERSIS dengan format berikut, tanpa tambahan apapun:
LABEL: negatif ATAU LABEL: bukan_negatif
ALASAN: <1 kalimat singkat>"""


def _build_fewshot_block(neg_examples, nonneg_examples) -> str:
    lines = [f'- "{t}" -> LABEL: negatif' for t in neg_examples]
    lines += [f'- "{t}" -> LABEL: bukan_negatif' for t in nonneg_examples]
    return "\n".join(lines)


def _parse_judge_response(text: str) -> tuple:
    label_match = re.search(r"LABEL:\s*(negatif|bukan_negatif)", text, re.IGNORECASE)
    alasan_match = re.search(r"ALASAN:\s*(.+)", text, re.IGNORECASE)
    label = label_match.group(1).lower() if label_match else "parse_gagal"
    alasan = alasan_match.group(1).strip() if alasan_match else text.strip()[:200]
    return label, alasan


def _judge_comment(comment: str, fewshot_block: str, retries: int = 3) -> tuple:
    prompt = JUDGE_PROMPT_TEMPLATE.format(fewshot=fewshot_block, comment=comment.replace('"', "'"))
    for attempt in range(retries):
        try:
            r = requests.post(f"{OLLAMA_HOST}/api/generate",
                               json={"model": JUDGE_MODEL, "prompt": prompt, "stream": False,
                                     "options": {"temperature": 0.0}}, timeout=60)
            r.raise_for_status()
            return _parse_judge_response(r.json()["response"])
        except Exception as e:
            if attempt == retries - 1:
                return "error", str(e)[:200]
            time.sleep(1.5 * (attempt + 1))

## STEP 1 - Semantic search (Layer 1) + LLM judge (Layer 2)\nBerat (bisa berjam-jam). Skip otomatis kalau `kandidat_final_untuk_review.csv` sudah ada.

In [ ]:
def search_negative_candidates() -> None:
    if KANDIDAT_FINAL.exists():
        print(f"[Layer1+2] Sudah ada -> {KANDIDAT_FINAL} (skip)")
        return

    if not DATASET_BINARY_BASELINE.exists():
        raise FileNotFoundError(f"{DATASET_BINARY_BASELINE} tidak ada. Jalankan tahap 09 dulu.")
    if not COMMENTS_RAW.exists():
        raise FileNotFoundError(f"{COMMENTS_RAW} tidak ada. Jalankan tahap 06 dulu.")

    labeled = pd.read_csv(DATASET_BINARY_BASELINE)
    ref_negatif_texts = labeled.loc[labeled["label_name"] == "negatif", "text_model"].dropna().unique().tolist()
    ref_nonneg_texts = labeled.loc[labeled["label_name"] != "negatif", "text_model"].dropna().tolist()
    print(f"[Layer1] Acuan negatif: {len(ref_negatif_texts)} komentar unik")

    ref_result = _embed_with_cache(ref_negatif_texts, EMBED_CACHE_REF, "Embedding acuan negatif")
    ref_embeds = np.vstack([ref_result[t] for t in ref_negatif_texts])

    raw = pd.read_csv(COMMENTS_RAW)
    raw = raw.dropna(subset=["comment_text"]).copy()
    raw["comment_text"] = raw["comment_text"].astype(str).str.strip()
    raw = raw[raw["comment_text"].str.len() > 0].reset_index(drop=True)

    already_labeled = set(labeled["comment_text_clean"].dropna().astype(str).str.strip())
    before = len(raw)
    raw = raw[~raw["comment_text"].isin(already_labeled)].reset_index(drop=True)
    print(f"[Layer1] Baris mentah: {before} -> {len(raw)} setelah exclude yang sudah dilabel")

    unique_texts = raw["comment_text"].unique().tolist()
    raw_result = _embed_with_cache(unique_texts, EMBED_CACHE_RAW, "Embedding comments.csv")

    valid_texts = [t for t in unique_texts if t in raw_result]
    emb_matrix = np.vstack([raw_result[t] for t in valid_texts])
    sims = _cosine_sim_matrix(emb_matrix, ref_embeds)
    max_sim = sims.max(axis=1)
    score_map = dict(zip(valid_texts, max_sim))

    raw["skor_kemiripan_negatif"] = raw["comment_text"].map(score_map).fillna(0.0)
    raw.sort_values("skor_kemiripan_negatif", ascending=False, inplace=True)
    raw.reset_index(drop=True, inplace=True)

    layer1 = raw.head(N_CANDIDATES).copy()
    layer1.to_csv(LAYER1_PATH, index=False)
    print(f"[Layer1] Selesai: {len(layer1)} kandidat -> {LAYER1_PATH}")

    rng = np.random.default_rng(SEED)
    fewshot_neg = rng.choice(ref_negatif_texts, size=min(N_FEWSHOT_EACH, len(ref_negatif_texts)), replace=False).tolist()
    fewshot_nonneg = rng.choice(ref_nonneg_texts, size=min(N_FEWSHOT_EACH, len(ref_nonneg_texts)), replace=False).tolist()
    fewshot_block = _build_fewshot_block(fewshot_neg, fewshot_nonneg)

    done_df = pd.DataFrame()
    done_ids = set()

    todo = layer1[~layer1["comment_id"].isin(done_ids)].copy()
    print(f"[Layer2] {len(todo)} kandidat perlu di-judge LLM...")

    results = []
    for i, row in enumerate(todo.itertuples(index=False), 1):
        label, alasan = _judge_comment(row.comment_text, fewshot_block)
        results.append({
            "comment_id": row.comment_id, "username": row.username, "video_id": row.video_id,
            "comment_text": row.comment_text, "skor_kemiripan_negatif": row.skor_kemiripan_negatif,
            "judge_label": label, "judge_alasan": alasan,
        })
        if i % CHECKPOINT_EVERY == 0:
            pd.concat([done_df, pd.DataFrame(results)], ignore_index=True).to_csv(KANDIDAT_FINAL, index=False)
            print(f"   [Layer2] {i}/{len(todo)}")

    final_df = pd.concat([done_df, pd.DataFrame(results)], ignore_index=True)
    final_df["urutan"] = final_df["judge_label"].map({"negatif": 0, "bukan_negatif": 1}).fillna(2)
    final_df.sort_values(["urutan", "skor_kemiripan_negatif"], ascending=[True, False], inplace=True)
    final_df.drop(columns="urutan", inplace=True)
    final_df.to_csv(KANDIDAT_FINAL, index=False)

    n_negatif = (final_df["judge_label"] == "negatif").sum()
    print(f"[Layer2] Selesai. LLM bilang NEGATIF: {n_negatif}/{len(final_df)} -> {KANDIDAT_FINAL}")
    print("[Layer2] Lanjut review manual (lihat markdown di atas).")


search_negative_candidates()

## [MANUAL] Review satu-per-satu lewat web app\nJalankan `python 02_labeling.py` (di luar notebook ini), buka `http://localhost:5050`, review tiap kandidat -> hasil tersimpan ke `review_manual.csv`. Lanjut ke cell berikutnya setelah review (boleh parsial).

## STEP 3 - Cross-check gabungan negatif (info saja)

In [ ]:
def merge_negative_candidates() -> None:
    if not REVIEW_MANUAL.exists():
        print(f"[Gabungan] {REVIEW_MANUAL} belum ada, skip (review manual belum dimulai).")
        return

    sumber = []
    if DATASET_BINARY_BASELINE.exists():
        base_df = pd.read_csv(DATASET_BINARY_BASELINE, dtype=str, keep_default_na=False, encoding="utf-8")
        label_col = next((c for c in ["label_name", "label_final", "label"] if c in base_df.columns), None)
        text_col = next((c for c in ["comment_text_clean", "text_model", "comment_text_raw"] if c in base_df.columns), None)
        if label_col and text_col:
            base_neg = base_df[base_df[label_col].astype(str).str.strip().str.lower() == "negatif"]
            base_neg = base_neg[base_neg[text_col].astype(str).str.strip() != ""]
            for i, row in base_neg.reset_index(drop=True).iterrows():
                sumber.append({"source": "baseline", "original_id": f"baseline_{i}", "original_text": row[text_col]})
            print(f"[Gabungan] Baseline: {len(base_neg)} komentar negatif")

    review_df = pd.read_csv(REVIEW_MANUAL, dtype=str, keep_default_na=False, encoding="utf-8")
    review_neg = review_df[review_df["label_manual"].astype(str).str.strip().str.lower() == "negatif"]
    review_neg = review_neg[review_neg["comment_text"].astype(str).str.strip() != ""]
    for _, row in review_neg.iterrows():
        cid = row.get("comment_id", "") or f"row{_}"
        sumber.append({"source": "review", "original_id": f"review_{cid}", "original_text": row["comment_text"]})
    print(f"[Gabungan] Review manual: {len(review_neg)} komentar negatif (dari {len(review_df)} baris direview)")

    sumber_df = pd.DataFrame(sumber).drop_duplicates(subset=["original_id"]).reset_index(drop=True)
    sumber_df.to_csv(KOMENTAR_NEGATIF_GABUNGAN, index=False)
    print(f"[Gabungan] Total: {len(sumber_df)} -> {KOMENTAR_NEGATIF_GABUNGAN}")


merge_negative_candidates()

## STEP 4 - Koreksi manual review_manual.csv (4 comment_id)

In [ ]:
def apply_review_corrections() -> None:
    if not REVIEW_MANUAL.exists():
        print("[Koreksi Review] review_manual.csv belum ada, skip.")
        return

    df = pd.read_csv(REVIEW_MANUAL, dtype=str, keep_default_na=False, encoding="utf-8")
    df["comment_id"] = df["comment_id"].astype(str)

    changed = []
    for cid, new_label in REVIEW_KOREKSI.items():
        mask = df["comment_id"] == cid
        if mask.sum() == 0 or (df.loc[mask, "label_manual"] == new_label).all():
            continue
        df.loc[mask, "label_manual"] = new_label
        changed.append(cid)

    if changed:
        shutil.copy2(REVIEW_MANUAL, REVIEW_MANUAL.with_name(REVIEW_MANUAL.name + ".bak"))
        df.to_csv(REVIEW_MANUAL, index=False, encoding="utf-8")
        print(f"[Koreksi Review] {len(changed)} baris dikoreksi: {changed}")
    else:
        print("[Koreksi Review] Tidak ada yang perlu dikoreksi (sudah diterapkan sebelumnya atau belum direview).")


apply_review_corrections()

## STEP 5 - Integrasikan negatif hasil review ke dataset_final.csv

In [ ]:
_URL_RE = re.compile(r"(https?://\S+|www\.\S+)", re.IGNORECASE)
_MENTION_RE = re.compile(r"@\w+")


def _clean_text_for_dataset(text: str) -> str:
    text = str(text)
    text = _URL_RE.sub("", text)
    text = _MENTION_RE.sub("", text)
    return re.sub(r"\s+", " ", text).strip()


def integrate_into_dataset() -> None:
    if not DATASET_FINAL.exists():
        if not DATASET_BINARY_BASELINE.exists():
            raise FileNotFoundError(f"{DATASET_BINARY_BASELINE} tidak ada. Jalankan tahap 09 dulu.")
        shutil.copy2(DATASET_BINARY_BASELINE, DATASET_FINAL)
        print(f"[Integrasi] Baseline dari tahap 09 disalin ke {DATASET_FINAL}")

    if not REVIEW_MANUAL.exists():
        print("[Integrasi] review_manual.csv belum ada, skip penambahan baris baru.")
        return

    df_dataset = pd.read_csv(DATASET_FINAL, dtype=str, keep_default_na=False, encoding="utf-8")
    df_review = pd.read_csv(REVIEW_MANUAL, dtype=str, keep_default_na=False, encoding="utf-8")

    if "comment_id" not in df_dataset.columns:
        df_dataset["comment_id"] = ""

    df_review_negatif = df_review[df_review["label_manual"] == "negatif"].copy()
    existing_ids = set(df_dataset.loc[df_dataset["comment_id"] != "", "comment_id"].astype(str))
    df_review_negatif = df_review_negatif[~df_review_negatif["comment_id"].astype(str).isin(existing_ids)]

    if len(df_review_negatif) == 0:
        print("[Integrasi] Tidak ada baris negatif baru yang perlu ditambahkan.")
        return

    baris_baru = pd.DataFrame({
        "username": df_review_negatif["username"], "video_id": df_review_negatif["video_id"],
        "comment_text_raw": df_review_negatif["comment_text"],
        "comment_text_clean": df_review_negatif["comment_text"].apply(_clean_text_for_dataset),
        "label_shafwa": "negatif", "label_khalila": "negatif", "label_final": "negatif",
        "label_source": "review_manual", "label_name": "negatif", "label": "1",
        "split": "", "fold": "", "comment_id": df_review_negatif["comment_id"],
    })
    baris_baru["text_model"] = baris_baru["comment_text_clean"]
    for col in df_dataset.columns:
        if col not in baris_baru.columns:
            baris_baru[col] = ""
    baris_baru = baris_baru[df_dataset.columns]

    df_gabung = pd.concat([df_dataset, baris_baru], ignore_index=True)
    shutil.copy2(DATASET_FINAL, DATASET_FINAL.with_name(DATASET_FINAL.name + ".bak"))
    df_gabung.to_csv(DATASET_FINAL, index=False, encoding="utf-8")
    print(f"[Integrasi] +{len(baris_baru)} baris baru (total: {len(df_gabung)}) -> {DATASET_FINAL}")


integrate_into_dataset()

## STEP 6 - Koreksi manual dataset_final.csv (3 teks + 1 substring)

In [ ]:
def apply_dataset_corrections() -> None:
    if not DATASET_FINAL.exists():
        print("[Koreksi Dataset] dataset_final.csv belum ada, skip.")
        return

    df = pd.read_csv(DATASET_FINAL, dtype=str, keep_default_na=False, encoding="utf-8")
    changed = []

    def _apply(mask, new_label_final, deskripsi):
        if mask.sum() == 0 or (df.loc[mask, "label_final"] == new_label_final).all():
            return
        df.loc[mask, "label_final"] = new_label_final
        df.loc[mask, "label_name"] = "non-negatif"
        df.loc[mask, "label"] = "0"
        changed.append(deskripsi)

    for teks, new_label_final in DATASET_KOREKSI_TEKS.items():
        mask = df["comment_text_clean"].astype(str).str.strip() == teks.strip()
        _apply(mask, new_label_final, teks[:50])

    mask_sub = df["comment_text_clean"].astype(str).str.contains(DATASET_KOREKSI_SUBSTRING, na=False)
    _apply(mask_sub, "netral", f"(substring) {DATASET_KOREKSI_SUBSTRING}")

    if changed:
        shutil.copy2(DATASET_FINAL, DATASET_FINAL.with_name(DATASET_FINAL.name + ".bak"))
        df.to_csv(DATASET_FINAL, index=False, encoding="utf-8")
        print(f"[Koreksi Dataset] {len(changed)} baris dikoreksi: {changed}")
    else:
        print("[Koreksi Dataset] Tidak ada yang perlu dikoreksi (sudah diterapkan sebelumnya).")


apply_dataset_corrections()

## STEP 7 - Re-split train/test + fold (seluruh dataset, dari nol)

In [ ]:
def split_dataset() -> None:
    if not DATASET_FINAL.exists():
        raise FileNotFoundError(f"{DATASET_FINAL} tidak ada.")

    df = pd.read_csv(DATASET_FINAL, dtype=str, keep_default_na=False, encoding="utf-8")

    train_idx, test_idx = train_test_split(
        df.index, test_size=TEST_SIZE, stratify=df["label_final"], random_state=RANDOM_SEED,
    )
    df["split"] = ""
    df.loc[train_idx, "split"] = "train"
    df.loc[test_idx, "split"] = "test"

    df["fold"] = ""
    df_train = df.loc[train_idx]
    min_class_count = df_train["label_final"].value_counts().min()
    effective_folds = min(N_FOLDS, int(min_class_count))
    if effective_folds < 2:
        raise ValueError("Kelas terkecil di train kurang dari 2 anggota, CV tidak bisa dijalankan.")

    skf = StratifiedKFold(n_splits=effective_folds, shuffle=True, random_state=RANDOM_SEED)
    for fold_num, (_, val_idx) in enumerate(skf.split(df_train.index, df_train["label_final"]), start=1):
        df.loc[df_train.index[val_idx], "fold"] = str(fold_num)

    shutil.copy2(DATASET_FINAL, DATASET_FINAL.with_name(DATASET_FINAL.name + ".bak"))
    df.to_csv(DATASET_FINAL, index=False, encoding="utf-8")

    print(f"[Split] Total: {len(df)} | Train: {len(train_idx)} ({100*len(train_idx)/len(df):.1f}%) | "
          f"Test: {len(test_idx)} ({100*len(test_idx)/len(df):.1f}%) | Fold: {effective_folds}")
    print("[Split] Proporsi label_final di train:")
    print(df[df["split"] == "train"]["label_final"].value_counts().to_string())
    print("[Split] Proporsi label_final di test:")
    print(df[df["split"] == "test"]["label_final"].value_counts().to_string())


split_dataset()

## STEP 8 - Ekstrak minoritas (negatif+positif) di TRAIN untuk sintesis

In [ ]:
def extract_minority_for_synthesis(output_csv: Path = MINORITAS_SINTESIS) -> None:
    if output_csv.exists():
        print(f"[Minoritas] Sudah ada -> {output_csv} (skip)")
        return

    df = pd.read_csv(DATASET_FINAL, dtype=str, keep_default_na=False, encoding="utf-8")
    if not (df["split"] == "train").any():
        raise ValueError("Kolom split belum keisi. Jalankan split_dataset() dulu.")

    catatan_skema = {"negatif": "3 kelas & binary", "positif": "3 kelas saja"}
    df_minoritas = df[(df["split"] == "train") & (df["label_final"].isin(["negatif", "positif"]))].copy()
    if len(df_minoritas) == 0:
        raise ValueError("Tidak ada baris negatif/positif di train.")

    cols_keep = [c for c in ["comment_id", "username", "video_id", "comment_text_clean", "label_source"]
                 if c in df_minoritas.columns]
    df_out = df_minoritas[cols_keep + ["label_final"]].reset_index(drop=True)
    df_out = df_out.rename(columns={"label_final": "kelas_asli"})
    df_out["catatan_skema"] = df_out["kelas_asli"].map(catatan_skema)
    df_out.insert(0, "row_num", df_out.index)

    df_out.to_csv(output_csv, index=False, encoding="utf-8")
    print(f"[Minoritas] {len(df_out)} baris (train only) -> {output_csv}")
    print(df_out["kelas_asli"].value_counts().to_string())


extract_minority_for_synthesis()